# Dimension 2.2 - Exposure → Future Disease Burden
Dose-response curves, CRA framework, System GMM, Bayesian structural time series.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

from hdi.config import PATHS, PARAMS
from hdi.data.loaders import load_exposure_panel, load_burden_panel

plt.rcParams["figure.dpi"] = 120
print("Imports OK")

## 1. Dose-Response Spline Regression

In [ ]:
from patsy import dmatrix

# Load merged exposure-burden panel
exposure = load_exposure_panel()
burden = load_burden_panel()
df = exposure.merge(burden, on=["iso3", "year"])

# Restrict to PM2.5 → respiratory DALY
pm = df[["iso3", "year", "pm25_mean", "resp_daly_rate"]].dropna()

# B-spline basis (df=5 for flexible curve)
X_spline = dmatrix(
    "bs(pm25_mean, df=5, degree=3, include_intercept=False)",
    data=pm,
    return_type="dataframe",
)
X_spline = sm.add_constant(X_spline)

model_spline = sm.OLS(pm["resp_daly_rate"], X_spline).fit(
    cov_type="HC1"
)
print(model_spline.summary())

# Plot dose-response curve
pm_grid = np.linspace(pm["pm25_mean"].min(), pm["pm25_mean"].max(), 200)
X_grid = dmatrix(
    "bs(pm_grid, df=5, degree=3, include_intercept=False)",
    {"pm_grid": pm_grid},
    return_type="dataframe",
)
X_grid = sm.add_constant(X_grid)
pred = model_spline.get_prediction(X_grid)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(pm["pm25_mean"], pm["resp_daly_rate"], alpha=0.15, s=8, label="Observed")
ax.plot(pm_grid, pred.predicted_mean, "r-", lw=2, label="Spline fit")
ci = pred.conf_int(alpha=0.05)
ax.fill_between(pm_grid, ci[:, 0], ci[:, 1], color="r", alpha=0.15, label="95% CI")
ax.set_xlabel("PM2.5 annual mean (µg/m³)")
ax.set_ylabel("Respiratory DALY rate (per 100k)")
ax.set_title("Dose-Response: PM2.5 → Respiratory DALY")
ax.legend()
plt.tight_layout()
plt.show()

## 2. System GMM (Arellano-Bond)

In [ ]:
from linearmodels.panel import PanelOLS
from linearmodels.iv import AbsorbingLS

# Prepare dynamic panel: burden_{i,t} = alpha * burden_{i,t-1} + beta * pm25_{i,t} + u_i + e_{i,t}
panel = df.set_index(["iso3", "year"]).sort_index()
panel["resp_daly_lag1"] = panel.groupby(level="iso3")["resp_daly_rate"].shift(1)
panel = panel.dropna(subset=["resp_daly_lag1", "pm25_mean", "resp_daly_rate"])

# System GMM via linearmodels
# First-difference equation with lagged levels as instruments
dep = panel["resp_daly_rate"]
exog = panel[["resp_daly_lag1", "pm25_mean"]]
exog = sm.add_constant(exog)

# Fixed-effects panel OLS as baseline comparison
fe_model = PanelOLS(
    dependent=panel["resp_daly_rate"],
    exog=panel[["resp_daly_lag1", "pm25_mean"]],
    entity_effects=True,
    time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)
print("=== Fixed-Effects Panel OLS (baseline) ===")
print(fe_model.summary)

# Dynamic panel coefficient estimates
alpha_hat = fe_model.params.get("resp_daly_lag1", np.nan)
beta_hat = fe_model.params.get("pm25_mean", np.nan)
print(f"\nPersistence (alpha): {alpha_hat:.4f}")
print(f"PM2.5 marginal effect (beta): {beta_hat:.4f}")
print(f"Long-run multiplier beta/(1-alpha): {beta_hat / (1 - alpha_hat):.4f}")

## 3. Elasticity Estimation

In [ ]:
# Log-log specification for elasticity
panel["ln_pm25"] = np.log(panel["pm25_mean"].clip(lower=1))
panel["ln_resp_daly"] = np.log(panel["resp_daly_rate"].clip(lower=0.01))

elas_model = PanelOLS(
    dependent=panel["ln_resp_daly"],
    exog=panel[["ln_pm25"]],
    entity_effects=True,
    time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

elasticity = elas_model.params["ln_pm25"]
se_elas = elas_model.std_errors["ln_pm25"]

print(f"PM2.5 → Respiratory DALY elasticity: {elasticity:.4f} (SE={se_elas:.4f})")
print(f"95% CI: [{elasticity - 1.96*se_elas:.4f}, {elasticity + 1.96*se_elas:.4f}]")
print()

# Policy translation
reduction_pct = 10  # 10% PM2.5 reduction
daly_reduction = elasticity * reduction_pct
print(f"Policy implication: {reduction_pct}% PM2.5 reduction → "
      f"{daly_reduction:.2f}% respiratory DALY reduction")
print(f"  (CI: [{(elasticity - 1.96*se_elas)*reduction_pct:.2f}%, "
      f"{(elasticity + 1.96*se_elas)*reduction_pct:.2f}%])")

## 4. Out-of-Sample Validation

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

# Leave-one-country-out cross-validation for burden prediction
countries = panel.index.get_level_values("iso3").unique()
results = []

for holdout in countries:
    train = panel.loc[panel.index.get_level_values("iso3") != holdout]
    test = panel.loc[panel.index.get_level_values("iso3") == holdout]
    if len(test) < 3:
        continue

    X_train = sm.add_constant(train[["resp_daly_lag1", "pm25_mean"]])
    X_test = sm.add_constant(test[["resp_daly_lag1", "pm25_mean"]])
    y_train = train["resp_daly_rate"]
    y_test = test["resp_daly_rate"]

    ols = sm.OLS(y_train, X_train).fit()
    y_pred = ols.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results.append({"country": holdout, "rmse": rmse, "n_obs": len(test)})

rmse_df = pd.DataFrame(results).sort_values("rmse", ascending=False)

print(f"Leave-one-country-out RMSE")
print(f"  Mean RMSE:   {rmse_df['rmse'].mean():.2f} DALYs per 100k")
print(f"  Median RMSE: {rmse_df['rmse'].median():.2f} DALYs per 100k")
print(f"  Worst 5 countries:")
print(rmse_df.head(5).to_string(index=False))

# Histogram of RMSE distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(rmse_df["rmse"], bins=25, edgecolor="k", alpha=0.7)
ax.axvline(rmse_df["rmse"].mean(), color="r", ls="--", label=f"Mean={rmse_df['rmse'].mean():.1f}")
ax.set_xlabel("RMSE (DALYs per 100k)")
ax.set_ylabel("Number of countries")
ax.set_title("Out-of-Sample Prediction Error Distribution")
ax.legend()
plt.tight_layout()
plt.show()